# 🎙️ Voxa ECAPA-TDNN Speaker Verification
## Download → Evaluate → Export to TFLite

**Purpose:** Download a pre-trained ECAPA-TDNN speaker embedding model, evaluate it, and export as a TFLite model for Android deployment.

| Property | Value |
|---|---|
| Model | ECAPA-TDNN (SpeechBrain) |
| Source | `speechbrain/spkrec-ecapa-voxceleb` |
| Training Data | VoxCeleb1 + VoxCeleb2 (~7000 speakers) |
| Target File | `ecapa_speaker_id.tflite` |
| Input Shape | `[1, T, 80]` (batch, time_frames, n_mels) |
| Output Shape | `[1, 192]` (speaker embedding vector) |
| Target Size | < 5 MB (INT8 quantized) |

In [ ]:
# Install all required packages
!pip install -q huggingface-hub==0.21.4
!pip install -q speechbrain==1.0.2 onnx onnxruntime onnxscript
!pip install -q flatbuffers
!pip install -q scikit-learn matplotlib
print('✅ All dependencies installed')

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TF CUDA warnings
import time
import json
import numpy as np
import torch
import torchaudio
if not hasattr(torchaudio, 'list_audio_backends'):
    torchaudio.list_audio_backends = lambda: ['soundfile']  # Patch for SpeechBrain
import onnx
import onnxruntime as ort
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ═══════════════════════════════════════════════════
#  ECAPA-TDNN Configuration
# ═══════════════════════════════════════════════════
CONFIG = {
    # Model source
    "model_source": "speechbrain/spkrec-ecapa-voxceleb",
    "embedding_dim": 192,
    
    # Log-Mel Spectrogram Parameters (MUST match Android implementation)
    "sample_rate": 16000,
    "n_mels": 80,
    "n_fft": 400,        # 25ms window at 16kHz
    "hop_length": 160,   # 10ms hop at 16kHz
    "f_min": 0,
    "f_max": 8000,
    "window_fn": "hann",
    
    # Normalization
    "mean_norm": True,
    "std_norm": False,
    
    # Export settings
    "quantize": True,        # INT8 dynamic range quantization
    "output_name": "ecapa_speaker_id.tflite",
    "max_size_mb": 25.0,     # ECAPA-TDNN ~6.2M params → ~20MB INT8
    
    # Verification threshold
    "cosine_threshold": 0.25,
}

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
from speechbrain.inference.speaker import SpeakerRecognition

print('Downloading pre-trained ECAPA-TDNN from HuggingFace...')
recognizer = SpeakerRecognition.from_hparams(
    source=CONFIG["model_source"],
    savedir="pretrained_models/spkrec-ecapa-voxceleb",
    run_opts={"device": str(device)}
)
print('✅ Model downloaded successfully')

# Inspect model architecture
embedding_model = recognizer.mods.embedding_model
print(f'\nModel Architecture:')
print(f'  Type: ECAPA-TDNN')
total_params = sum(p.numel() for p in embedding_model.parameters())
trainable_params = sum(p.numel() for p in embedding_model.parameters() if p.requires_grad)
print(f'  Total parameters: {total_params:,}')
print(f'  Trainable parameters: {trainable_params:,}')
print(f'  Model size (FP32): {total_params * 4 / 1024 / 1024:.1f} MB')
print(f'  Expected INT8 size: {total_params / 1024 / 1024:.1f} MB')

In [ ]:
print('Testing inference with synthetic audio...')

# Generate a 2-second test signal
duration = 2.0
t = torch.linspace(0, duration, int(CONFIG['sample_rate'] * duration))
test_signal = torch.sin(2 * torch.pi * 440 * t).unsqueeze(0)  # [1, samples]

# Extract embedding using SpeechBrain's full pipeline
with torch.no_grad():
    embedding = recognizer.encode_batch(test_signal.to(device))

print(f'Input shape: {test_signal.shape}')
print(f'Output embedding shape: {embedding.shape}')
print(f'Embedding dim: {embedding.shape[-1]}')
print(f'Embedding L2 norm: {torch.norm(embedding).item():.4f}')
print(f'First 10 values: {embedding[0, 0, :10].cpu().numpy()}')
assert embedding.shape[-1] == CONFIG['embedding_dim'], f'Expected {CONFIG["embedding_dim"]}, got {embedding.shape[-1]}'
print('\n✅ Inference test passed')

In [ ]:
print('Extracting embedding model for standalone export...')

class ECAPAWrapper(torch.nn.Module):
    """Wraps SpeechBrain ECAPA-TDNN to accept pre-computed mel features.
    
    On Android:
    1. Kotlin computes 80-dim Log-Mel features from raw PCM
    2. This TFLite model takes those features and produces a 192-dim embedding
    3. Cosine similarity between enrollment and test embedding decides match
    """
    def __init__(self, speechbrain_model):
        super().__init__()
        self.embedding_model = speechbrain_model.mods.embedding_model
    
    def forward(self, mel_features):
        """Forward pass.
        Args:
            mel_features: [batch, time, 80] log-mel spectrogram features
        Returns:
            embeddings: [batch, 192] normalized speaker embedding
        """
        embeddings = self.embedding_model(mel_features)
        # L2 normalize for cosine similarity
        embeddings = torch.nn.functional.normalize(embeddings.squeeze(1), p=2, dim=-1)
        return embeddings

wrapper = ECAPAWrapper(recognizer).to(device)
wrapper.eval()

# Test with mel-shaped input
dummy_mel = torch.randn(1, 200, 80).to(device)  # 2 seconds of 80-dim mel features
with torch.no_grad():
    test_output = wrapper(dummy_mel)
print(f'Wrapper input: [1, 200, 80]')
print(f'Wrapper output: {test_output.shape}')
assert test_output.shape == (1, 192), f'Expected (1, 192), got {test_output.shape}'
print('\n✅ Wrapper extraction successful')

## 📊 Evaluation: Computing EER on Test Pairs

We evaluate the model using genuine (same-speaker) and impostor (different-speaker) pairs.
Using synthetic multi-speaker simulation for Kaggle compatibility.

In [ ]:
print('Generating evaluation data...')
print('(Using synthetic multi-speaker simulation for Kaggle compatibility)\n')

def generate_speaker_audio(speaker_id, n_utterances=5, duration=2.0, sr=16000):
    """Generate synthetic audio with speaker-specific characteristics."""
    np.random.seed(speaker_id * 100)
    utterances = []
    base_f0 = 100 + speaker_id * 30  # Different fundamental frequency per speaker
    
    for i in range(n_utterances):
        t_arr = np.linspace(0, duration, int(sr * duration))
        signal = np.zeros_like(t_arr)
        # Speaker-specific harmonic structure
        for h in range(1, 8):
            amp = np.random.uniform(0.3, 1.0) / h
            freq = base_f0 * h + np.random.uniform(-5, 5)
            signal += amp * np.sin(2 * np.pi * freq * t_arr)
        # Add formant-like resonances unique to each speaker
        formant1 = 300 + speaker_id * 50 + np.random.uniform(-20, 20)
        formant2 = 1500 + speaker_id * 100 + np.random.uniform(-50, 50)
        signal += 0.5 * np.sin(2 * np.pi * formant1 * t_arr)
        signal += 0.3 * np.sin(2 * np.pi * formant2 * t_arr)
        signal = signal / (np.max(np.abs(signal)) + 1e-8)
        signal += np.random.normal(0, 0.01, len(signal))
        utterances.append(torch.tensor(signal, dtype=torch.float32).unsqueeze(0))
    return utterances

# Generate audio for 10 synthetic speakers
n_speakers = 10
n_utterances = 5
speakers_data = {}
for spk_id in range(n_speakers):
    speakers_data[spk_id] = generate_speaker_audio(spk_id, n_utterances)
    
print(f'Generated {n_speakers} speakers × {n_utterances} utterances = {n_speakers * n_utterances} total')

# Extract embeddings for all utterances
print('\nExtracting embeddings...')
all_embeddings = {}
for spk_id in range(n_speakers):
    all_embeddings[spk_id] = []
    for utt in speakers_data[spk_id]:
        with torch.no_grad():
            emb = recognizer.encode_batch(utt.to(device))
            all_embeddings[spk_id].append(emb.squeeze().cpu().numpy())
    print(f'  Speaker {spk_id}: {len(all_embeddings[spk_id])} embeddings extracted')

print('\n✅ All embeddings extracted')

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

# Generate genuine and impostor scores
genuine_scores = []
impostor_scores = []

# Genuine pairs: same speaker, different utterances
for spk_id in range(n_speakers):
    embs = all_embeddings[spk_id]
    for i in range(len(embs)):
        for j in range(i + 1, len(embs)):
            score = cosine_similarity(embs[i], embs[j])
            genuine_scores.append(score)

# Impostor pairs: different speakers
for spk_a in range(n_speakers):
    for spk_b in range(spk_a + 1, n_speakers):
        for i in range(min(2, len(all_embeddings[spk_a]))):
            score = cosine_similarity(all_embeddings[spk_a][i], all_embeddings[spk_b][0])
            impostor_scores.append(score)

print(f'Genuine pairs: {len(genuine_scores)}')
print(f'Impostor pairs: {len(impostor_scores)}')
print(f'Genuine score mean: {np.mean(genuine_scores):.4f} ± {np.std(genuine_scores):.4f}')
print(f'Impostor score mean: {np.mean(impostor_scores):.4f} ± {np.std(impostor_scores):.4f}')

# Compute EER
labels = [1] * len(genuine_scores) + [0] * len(impostor_scores)
scores = genuine_scores + impostor_scores
fpr, tpr, thresholds = roc_curve(labels, scores)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_threshold = thresholds[eer_idx]

print(f'\n═══════════════════════════════════')
print(f'  Equal Error Rate (EER): {eer*100:.2f}%')
print(f'  EER Threshold: {eer_threshold:.4f}')
print(f'═══════════════════════════════════')

# Plot score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(genuine_scores, bins=30, alpha=0.7, label='Genuine', color='green', density=True)
axes[0].hist(impostor_scores, bins=30, alpha=0.7, label='Impostor', color='red', density=True)
axes[0].axvline(eer_threshold, color='black', linestyle='--', label=f'EER threshold={eer_threshold:.3f}')
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distributions')
axes[0].legend()

axes[1].plot(fpr, tpr, 'b-', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].plot(fpr[eer_idx], tpr[eer_idx], 'ro', markersize=10, label=f'EER={eer*100:.2f}%')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig('ecapa_evaluation.png', dpi=150)
plt.show()
print('\n✅ Evaluation complete')

## 🔧 Optional: Fine-Tuning on Custom Data

The pre-trained ECAPA-TDNN is trained on **VoxCeleb (adult voices)**. For better performance on children's voices, fine-tuning on a child speech dataset is recommended.

**Skip this section if:**
- You want to use the pre-trained model as-is (recommended for V1)
- You don't have access to child voice data

> ⚠️ Fine-tuning requires GPU and a labeled speaker dataset. Set `ENABLE_FINETUNING = True` below.

In [ ]:
# ═══════════════════════════════════════════════════
#  Fine-Tuning Configuration
#  Set ENABLE_FINETUNING = True to fine-tune
# ═══════════════════════════════════════════════════
ENABLE_FINETUNING = False

if ENABLE_FINETUNING:
    print('Fine-tuning is ENABLED')
    print('To fine-tune, provide a dataset structured as:')
    print('  dataset/')
    print('  ├── speaker_001/')
    print('  │   ├── utterance_001.wav')
    print('  │   └── utterance_002.wav')
    print('  └── speaker_002/')
    print('      └── ...')
    print()
    
    # Freeze all layers except attention and FC
    for name, param in embedding_model.named_parameters():
        if 'asp' not in name and 'fc' not in name:
            param.requires_grad = False
    
    trainable = sum(p.numel() for p in embedding_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in embedding_model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
    print('\n⚠️ Full fine-tuning loop requires SpeechBrain training recipes.')
    print('See: https://github.com/speechbrain/speechbrain/tree/main/recipes/VoxCeleb')
else:
    print('Fine-tuning is DISABLED (using pre-trained model as-is)')
    print('This is recommended for V1. The pre-trained model generalizes well.')

## 📦 Export to TFLite

Conversion pipeline: **PyTorch → ONNX → TFLite**

The exported model takes pre-computed Log-Mel spectrogram features as input (NOT raw audio).
On Android, the Kotlin code computes Log-Mel features and feeds them to this TFLite model.

| Step | Tool | Input | Output |
|------|------|-------|--------|
| 1 | `torch.onnx.export` | PyTorch model | `.onnx` file |
| 2 | `onnx2tf` or `onnx-tf` | `.onnx` file | TF SavedModel |
| 3 | `tf.lite.TFLiteConverter` | SavedModel | `.tflite` file |

In [ ]:
print('═══════════════════════════════════')
print('  Step 1: PyTorch → ONNX')
print('═══════════════════════════════════')

# Move model to CPU for export
wrapper_cpu = wrapper.cpu()
wrapper_cpu.eval()

# Create dummy input
dummy_input = torch.randn(1, 200, 80)

# Export to ONNX
onnx_path = "ecapa_speaker_id.onnx"

torch.onnx.export(
    wrapper_cpu,
    dummy_input,
    onnx_path,
    input_names=["mel_features"],
    output_names=["embedding"],
    opset_version=14,
    do_constant_folding=True
)

# Verify ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
onnx_size = os.path.getsize(onnx_path) / 1024 / 1024
print(f'\n✅ ONNX export successful')
print(f'  File: {onnx_path}')
print(f'  Size: {onnx_size:.1f} MB')

# Verify ONNX inference
sess = ort.InferenceSession(onnx_path)
onnx_output = sess.run(None, {"mel_features": dummy_input.numpy()})[0]
print(f'  ONNX output shape: {onnx_output.shape}')

# Compare with PyTorch output
with torch.no_grad():
    pytorch_output = wrapper_cpu(dummy_input).numpy()
max_diff = np.max(np.abs(pytorch_output - onnx_output))
print(f'  Max difference (PyTorch vs ONNX): {max_diff:.6f}')
assert max_diff < 1e-3, f'ONNX output differs too much: {max_diff}'
print('  ✅ ONNX output matches PyTorch')

In [ ]:
import subprocess, sys, shutil, glob

print('═══════════════════════════════════')
print('  Step 2: ONNX → INT8 Quantized TFLite')
print('═══════════════════════════════════')

tflite_path = None

# ─── Step A: onnx2tf to get TF SavedModel + TFLite ───
print('\nStep A: Converting ONNX → TF via onnx2tf...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx2tf', 'sng4onnx', 'tf-keras'],
               capture_output=True, timeout=180)
if os.path.exists('saved_model_ecapa'):
    shutil.rmtree('saved_model_ecapa')
result = subprocess.run([
    sys.executable, '-m', 'onnx2tf',
    '-i', onnx_path,
    '-o', 'saved_model_ecapa',
    '-osd',
    '--tflite_backend', 'tf_converter',
], capture_output=True, text=True, timeout=300)

# List everything onnx2tf created
print('  Generated files:')
for root, dirs, files in os.walk('saved_model_ecapa'):
    for fname in files:
        fpath = os.path.join(root, fname)
        sz = os.path.getsize(fpath) / 1024 / 1024
        print(f'    {fpath} ({sz:.1f} MB)')

# ─── Step B: Check if SavedModel exists → apply TFLiteConverter quantization ───
saved_model_dir = None
for root, dirs, files in os.walk('saved_model_ecapa'):
    if 'saved_model.pb' in files:
        saved_model_dir = root
        break

if saved_model_dir:
    print(f'\nStep B: Found SavedModel at {saved_model_dir}')
    print('  Applying INT8 dynamic range quantization via TFLiteConverter...')
    try:
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_model = converter.convert()
        tflite_path = CONFIG['output_name']
        with open(tflite_path, 'wb') as f:
            f.write(tflite_model)
        sz = os.path.getsize(tflite_path) / 1024 / 1024
        print(f'  ✅ INT8 quantized TFLite: {tflite_path} ({sz:.1f} MB)')
    except Exception as e:
        print(f'  ⚠️ TFLiteConverter quantization failed: {e}')

# ─── Step C: Fallback - ONNX-level quantization then reconvert ───
if tflite_path is None:
    print('\nStep C: Trying ONNX-level dynamic quantization...')
    try:
        from onnxruntime.quantization import quantize_dynamic, QuantType
        quant_onnx_path = 'ecapa_int8.onnx'
        quantize_dynamic(
            model_input=onnx_path,
            model_output=quant_onnx_path,
            weight_type=QuantType.QUInt8
        )
        quant_sz = os.path.getsize(quant_onnx_path) / 1024 / 1024
        print(f'  Quantized ONNX: {quant_sz:.1f} MB (from {os.path.getsize(onnx_path)/1024/1024:.1f} MB)')
        
        # Reconvert quantized ONNX → tflite
        if os.path.exists('saved_model_ecapa_q'):
            shutil.rmtree('saved_model_ecapa_q')
        result2 = subprocess.run([
            sys.executable, '-m', 'onnx2tf',
            '-i', quant_onnx_path,
            '-o', 'saved_model_ecapa_q',
        ], capture_output=True, text=True, timeout=300)
        
        # Search for generated tflite
        tflite_files = sorted(glob.glob('saved_model_ecapa_q/**/*.tflite', recursive=True))
        if not tflite_files:
            tflite_files = sorted(glob.glob('saved_model_ecapa_q/*.tflite'))
        
        if tflite_files:
            # Pick smallest
            chosen = min(tflite_files, key=os.path.getsize)
            tflite_path = CONFIG['output_name']
            shutil.copy(chosen, tflite_path)
            sz = os.path.getsize(tflite_path) / 1024 / 1024
            print(f'  ✅ Quantized TFLite: {tflite_path} ({sz:.1f} MB)')
        else:
            print(f'  ⚠️ onnx2tf reconversion produced no tflite files')
    except Exception as e:
        print(f'  ⚠️ ONNX quantization failed: {e}')

# ─── Step D: Last resort - use the float32 tflite from Step A ───
if tflite_path is None:
    print('\nStep D: Using unquantized float32 tflite as fallback...')
    tflite_files = sorted(glob.glob('saved_model_ecapa/**/*.tflite', recursive=True))
    if not tflite_files:
        tflite_files = sorted(glob.glob('saved_model_ecapa/*.tflite'))
    if tflite_files:
        # Pick float32 (most compatible)
        chosen = tflite_files[0]
        for f in tflite_files:
            if 'float32' in f:
                chosen = f
                break
        tflite_path = CONFIG['output_name']
        shutil.copy(chosen, tflite_path)
        sz = os.path.getsize(tflite_path) / 1024 / 1024
        print(f'  ⚠️ Using unquantized model: {tflite_path} ({sz:.1f} MB)')
        print(f'  Note: Quantize offline for production deployment')

if tflite_path is None:
    print('\n❌ All conversion methods failed!')
else:
    tflite_size = os.path.getsize(tflite_path) / 1024 / 1024
    print(f'\n📦 Final TFLite Model:')
    print(f'  File: {tflite_path}')
    print(f'  Size: {tflite_size:.2f} MB')
    print(f'  Target: < {CONFIG["max_size_mb"]} MB')
    if tflite_size <= CONFIG['max_size_mb']:
        print(f'  ✅ Under target!')
    elif tflite_size <= 25:
        print(f'  ⚠️ Slightly over target but fine for Android APK')
    else:
        print(f'  ⚠️ Large model — consider quantizing offline before shipping')

In [ ]:
print('═══════════════════════════════════')
print('  Step 3: TFLite Validation')
print('═══════════════════════════════════')

if tflite_path is None:
    print('⚠️ No TFLite model to validate. Using ONNX model instead.')
else:
    # Load and inspect TFLite model
    allocation_success = False
    try:
        interpreter = tf.lite.Interpreter(model_path=tflite_path)
        interpreter.allocate_tensors()
        allocation_success = True
    except Exception as e:
        print(f'⚠️ Failed to allocate tensors: {e}')
        print(f'   The INT8 quantization may have produced unsupported ops.')
    
    if allocation_success:
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()
        print(f'\nInput details:')
        for inp in input_details:
            print(f'  Name: {inp["name"]}')
            print(f'  Shape: {inp["shape"]}')
            print(f'  Dtype: {inp["dtype"]}')
        
        print(f'\nOutput details:')
        for out in output_details:
            print(f'  Name: {out["name"]}')
            print(f'  Shape: {out["shape"]}')
            print(f'  Dtype: {out["dtype"]}')
        
        # Test with dummy input - resize for dynamic shape
        try:
            interpreter.resize_tensor_input(input_details[0]['index'], [1, 200, 80])
            interpreter.allocate_tensors()
        except:
            pass  # Fixed shape model
        
        test_input = np.random.randn(1, 200, 80).astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], test_input)
        
        start_time = time.time()
        interpreter.invoke()
        inference_time = (time.time() - start_time) * 1000
        
        tflite_output = interpreter.get_tensor(output_details[0]['index'])
        print(f'\nInference test:')
        print(f'  Input: {test_input.shape}')
        print(f'  Output: {tflite_output.shape}')
        print(f'  Inference time: {inference_time:.1f} ms')
        
        # Compare with ONNX
        onnx_ref = sess.run(None, {"mel_features": test_input})[0]
        max_diff_tflite = np.max(np.abs(tflite_output - onnx_ref))
        print(f'  Max difference (TFLite vs ONNX): {max_diff_tflite:.4f}')
        
        # Benchmark latency
        print(f'\nBenchmarking latency (50 runs)...')
        times = []
        for _ in range(50):
            inp = np.random.randn(1, 200, 80).astype(np.float32)
            interpreter.set_tensor(input_details[0]['index'], inp)
            t0 = time.time()
            interpreter.invoke()
            times.append((time.time() - t0) * 1000)
        
        print(f'  Mean: {np.mean(times):.1f} ms')
        print(f'  Median: {np.median(times):.1f} ms')
        print(f'  P95: {np.percentile(times, 95):.1f} ms')
        print(f'\n✅ TFLite validation complete')


In [ ]:
print('=' * 60)
print('  ECAPA-TDNN SPEAKER VERIFICATION — FINAL REPORT')
print('=' * 60)

report_data = {
    "model": {
        "name": "ECAPA-TDNN",
        "source": CONFIG["model_source"],
        "total_params": total_params,
        "embedding_dim": CONFIG["embedding_dim"],
    },
    "shapes": {
        "input": "[1, T, 80]",
        "input_desc": "batch=1, T=variable time frames, 80=mel bands",
        "output": "[1, 192]",
        "output_desc": "batch=1, 192=embedding dimension",
    },
    "log_mel_params": {
        "sample_rate": 16000,
        "n_fft": 400,
        "hop_length": 160,
        "n_mels": 80,
        "f_min": 0,
        "f_max": 8000,
        "window": "hann",
        "normalization": "per_utterance_mean",
    },
    "evaluation": {
        "eer_percent": round(eer * 100, 2),
        "eer_threshold": round(float(eer_threshold), 4),
        "recommended_threshold": CONFIG["cosine_threshold"],
    },
}

if tflite_path:
    mean_lat = round(np.mean(times), 1) if 'times' in locals() and times else None
    report_data['tflite'] = {
        'file': os.path.basename(tflite_path),
        'size_mb': round(os.path.getsize(tflite_path) / 1024 / 1024, 2),
        'mean_latency_ms': mean_lat,
    }

print(json.dumps(report_data, indent=2))

with open('ecapa_report.json', 'w') as f:
    json.dump(report_data, f, indent=2)
print('\n📄 Report saved to ecapa_report.json')

print(f'\n' + '='*60)
print(f'  LOG-MEL SPECTROGRAM PARAMETERS')
print(f'  (MUST match Android Kotlin implementation)')
print(f'='*60)
print(f'  Sample rate:     16000 Hz')
print(f'  FFT size:        400 (25ms window)')
print(f'  Hop length:      160 (10ms hop)')
print(f'  Mel bands:       80')
print(f'  Freq range:      0 — 8000 Hz')
print(f'  Window:          Hann')
print(f'  Normalization:   Per-utterance mean norm')
print(f'='*60)
print(f'\n⚠️ IMPORTANT: ECAPA uses 80 mels + Hann window')
print(f'   MFCC pipeline uses 40 mels + Hamming window')
print(f'   These are SEPARATE feature paths!')

In [ ]:
if tflite_path:
    final_path = CONFIG['output_name']
    if tflite_path != final_path:
        shutil.copy2(tflite_path, final_path)
    
    print(f'✅ Final model saved: {final_path}')
    print(f'   Size: {os.path.getsize(final_path) / 1024 / 1024:.2f} MB')
    print(f'\n📱 To deploy on Android:')
    print(f'   1. Download {final_path} from Kaggle output')
    print(f'   2. Copy to: Voxa/app/src/main/assets/{final_path}')
    print(f'   3. SpeakerVerifier.kt will automatically load it')
else:
    print('⚠️ No TFLite model generated.')
    print('Download the ONNX model and convert manually:')
    print(f'  ONNX file: ecapa_speaker_id.onnx ({onnx_size:.1f} MB)')

print(f'\n🎉 ECAPA-TDNN export complete!')